# CONFIG

In [131]:
import polars as pl
import sqlite3
import tomllib
import glob
import numpy as np

In [2]:
from datetime import datetime
import os

In [3]:
import paramiko
from scp import SCPClient

In [4]:
with open("secrets.toml", "rb") as f:
    config = tomllib.load(f)
creds = config["mikrus"]

In [5]:
TABLES = ["offers", "offers_tytan", "offers_gralczyk", "offers_polnoc", "analyzed_offers" ]

In [6]:
pl.Config.set_tbl_rows(10)
pl.Config.STR_SET_ELIDED_BOUND = 500 
pl.Config.TT_CHOP_STR_BOUND = 500     
pl.Config.TABLE_WIDTH = 1000     
pl.Config.set_fmt_str_lengths(1000)

polars.config.Config

In [7]:
CITIES_GEO = {
    'ELŻBIECIN': {'lat': 53.1672, 'lon': 22.1231},
    'ZAMBRÓW': {'lat': 52.9856, 'lon': 22.2428},
    'STARE ZAKRZEWO': {'lat': 53.0042, 'lon': 22.2983},
    'PUPKI': {'lat': 53.2258, 'lon': 21.8497},
    'WYGODA': {'lat': 53.1022, 'lon': 22.1481},
    'STARE MODZELE': {'lat': 53.1114, 'lon': 22.3164},
    'NOWOGRÓD': {'lat': 53.2264, 'lon': 21.8806},
    'CZERWONE': {'lat': 53.4336, 'lon': 21.9044},
    'JANKI STARE': {'lat': 52.9419, 'lon': 22.4278},
    'GRĄDY': {'lat': 53.1158, 'lon': 22.1158},
    'CZACHY': {'lat': 53.0453, 'lon': 22.3456},
    'GROCHY': {'lat': 52.9461, 'lon': 22.3614},
    'DROZDOWO': {'lat': 53.1492, 'lon': 22.1644},
    'DĄBEK': {'lat': 53.0414, 'lon': 21.4647},
    'OSTROŁĘKA': {'lat': 53.0842, 'lon': 21.5731},
    'SIEMIEŃ NADRZECZNY': {'lat': 53.1978, 'lon': 22.1853},
    'SZLASY': {'lat': 53.0614, 'lon': 22.0647},
    'GRZYMAŁY SZCZEPANKOWSKIE': {'lat': 53.1436, 'lon': 21.9892},
    'CZERWIN': {'lat': 52.9239, 'lon': 21.6442},
    'FILOCHY': {'lat': 52.9481, 'lon': 21.7258},
    'KISIOŁKI': {'lat': 53.1667, 'lon': 22.3333},
    'BALIKI': {'lat': 53.2847, 'lon': 21.8594},
    'JANOWO': {'lat': 53.4864, 'lon': 21.9069},
    'SZUMOWO': {'lat': 52.9153, 'lon': 22.0736},
    'STARA ŁOMŻA PRZY SZOSIE': {'lat': 53.1603, 'lon': 22.0944},
    'NOWY CYDZYN': {'lat': 53.2753, 'lon': 22.1764},
    'STARE BOŻEJEWO': {'lat': 53.2383, 'lon': 22.3486},
    'DŁUGI KĄT': {'lat': 53.3308, 'lon': 21.7375},
    'ZAWADY': {'lat': 53.1539, 'lon': 22.6653},
    'MIASTKOWO': {'lat': 53.1489, 'lon': 21.8211},
    'KSIĘŻOPOLE': {'lat': 52.3306, 'lon': 22.2547},
    'JEDWABNE': {'lat': 53.2861, 'lon': 22.3019},
    'NOWE KUPISKI': {'lat': 53.1975, 'lon': 21.9844},
    'TRUSZKI': {'lat': 53.3236, 'lon': 22.1158},
    'ŚWIDRY': {'lat': 53.1561, 'lon': 21.9056},
    'SASINY': {'lat': 52.6186, 'lon': 22.9511},
    'WYSOKIE MAŁE': {'lat': 53.2681, 'lon': 22.1156},
    'WIZNA': {'lat': 53.1931, 'lon': 22.3831},
    'RUTKI': {'lat': 53.1000, 'lon': 22.4167},
    'SIEMNOCHA': {'lat': 53.2458, 'lon': 22.0833},
    'NAGÓRKI': {'lat': 53.1581, 'lon': 22.3333},
    'CZAPLICE': {'lat': 53.1114, 'lon': 22.1258},
    'MROCZKI': {'lat': 52.9983, 'lon': 22.3364},
    'JURZEC WŁOŚCIAŃSKI': {'lat': 53.3000, 'lon': 22.2167},
    'KOLNO': {'lat': 53.4131, 'lon': 21.9322},
    'STARE RATOWO': {'lat': 53.0114, 'lon': 21.8456},
    'KRUSZA': {'lat': 53.0781, 'lon': 21.5031},
    'GRODZISK DUŻY': {'lat': 52.4833, 'lon': 22.7500},
    'OJCEWO': {'lat': 53.3364, 'lon': 22.2542},
    'BACZE SUCHE': {'lat': 53.0564, 'lon': 22.1647},
    'GOSIE MAŁE': {'lat': 53.0244, 'lon': 22.4214},
    'JABŁONKA': {'lat': 53.5358, 'lon': 21.5061},
    'POPIOŁKI': {'lat': 53.3642, 'lon': 21.8597},
    'DRWĘCZ': {'lat': 53.0647, 'lon': 21.6642},
    'PODGÓRZE': {'lat': 53.1203, 'lon': 22.1236},
    'RZEKUŃ': {'lat': 53.0561, 'lon': 21.6167},
    'STARA ŁOMŻA NAD RZEKĄ': {'lat': 53.1656, 'lon': 22.1128},
    'SZCZEPANKOWO': {'lat': 53.1167, 'lon': 21.9833},
    'JARNUTY': {'lat': 53.1167, 'lon': 21.6833},
    'TUROŚL': {'lat': 53.3853, 'lon': 21.7258},
    'STARE KUPISKI': {'lat': 53.1972, 'lon': 21.9964},
    'KUPISKI STARE': {'lat': 53.1972, 'lon': 21.9964},
    'DŁUGOBÓRZ PIERWSZY': {'lat': 52.9667, 'lon': 22.2167},
    'WIKTORZYN': {'lat': 52.8833, 'lon': 21.7500},
    'KACZYNEK': {'lat': 53.0114, 'lon': 21.6831},
    'RYBAKI': {'lat': 53.1842, 'lon': 22.3164},
    'SAMBORY': {'lat': 53.2258, 'lon': 22.3853},
    'LELIS': {'lat': 53.1592, 'lon': 21.5644},
    'RUDKA': {'lat': 52.7333, 'lon': 22.7333},
    'KRAJEWO': {'lat': 52.9833, 'lon': 22.3333},
    'WYŁUDZIN': {'lat': 53.2458, 'lon': 21.7258},
    'LASKI SZLACHECKIE': {'lat': 52.9414, 'lon': 21.4647},
    'RYDZEWO': {'lat': 53.3086, 'lon': 22.0647},
    'SULĘCIN SZLACHECKI': {'lat': 52.8464, 'lon': 21.9167},
    'PIĄTNICA PODUCHOWNA': {'lat': 53.1894, 'lon': 22.0919},
    'BARTKI': {'lat': 53.3414, 'lon': 21.5258},
    'KORYTKI LEŚNE': {'lat': 53.1333, 'lon': 21.9500},
    'OSTROŻNE': {'lat': 52.9833, 'lon': 22.1167},
    'PROSIENICA': {'lat': 52.9167, 'lon': 21.9500},
    'STARY CYDZYN': {'lat': 53.2753, 'lon': 22.1856},
    'JURKI': {'lat': 53.3983, 'lon': 22.1114},
    'DOBRY LAS': {'lat': 53.3167, 'lon': 21.8333},
    'LACHOWO': {'lat': 53.4667, 'lon': 22.0167},
    'GOSTERY': {'lat': 53.0333, 'lon': 21.8667},
    'STARE KRAJEWO': {'lat': 52.9833, 'lon': 22.3500},
    'STARY LUBOTYŃ': {'lat': 52.9167, 'lon': 21.8833},
    'BORKOWO': {'lat': 53.2986, 'lon': 21.9142},
    'MOCARZE': {'lat': 53.2758, 'lon': 22.4258},
    'TABĘDZ': {'lat': 53.0333, 'lon': 22.1667},
    'JEZIORKO': {'lat': 53.2281, 'lon': 22.1456},
    'ŁAWY': {'lat': 53.0667, 'lon': 21.6167},
    'ŁOCHTYNOWO': {'lat': 53.1667, 'lon': 22.0333},
    'WOLA ZAMBROWSKA': {'lat': 52.9667, 'lon': 22.2500},
    'LASKOWIEC': {'lat': 53.0833, 'lon': 21.5333},
    'GIEŁCZYN': {'lat': 53.1114, 'lon': 22.0542},
    'PORYTE': {'lat': 53.3258, 'lon': 22.0831},
    'ŁOMŻA': {'lat': 53.1781, 'lon': 22.0592},
    'TROSZYN': {'lat': 53.0333, 'lon': 21.7333},
    'MOTYKA': {'lat': 53.1667, 'lon': 21.8833},
    'PIANKI': {'lat': 53.2167, 'lon': 22.0167},
    'KISIELNICA': {'lat': 53.2286, 'lon': 22.0542},
    'KONOPKI': {'lat': 53.0667, 'lon': 22.3167},
    'BURZYN': {'lat': 53.2758, 'lon': 22.4647},
    'KOSAKI': {'lat': 53.1667, 'lon': 22.2833},
    'KĄTY': {'lat': 53.3167, 'lon': 22.1667},
    'RATOWO': {'lat': 53.0167, 'lon': 21.8333},
    'PNIEWO': {'lat': 53.1167, 'lon': 22.0833},
    'KOŁAKI KOŚCIELNE': {'lat': 53.0167, 'lon': 22.3667},
    'BUDY CZARNOCKIE': {'lat': 53.2167, 'lon': 22.1500},
    'KOWNATY': {'lat': 53.2667, 'lon': 22.2167},
    'OSETNO': {'lat': 53.6833, 'lon': 19.8667},
    'MIKOŁAJKI': {'lat': 53.8014, 'lon': 21.5714},
    'KONARZYCE': {'lat': 53.1333, 'lon': 22.0500},
    'JEDNACZEWO': {'lat': 53.2081, 'lon': 21.9542},
    'PRZYTUŁY': {'lat': 53.3667, 'lon': 22.3167},
    'GOŁASZE': {'lat': 52.9833, 'lon': 22.5000},
    'KRZEWO': {'lat': 53.1500, 'lon': 22.3667},
    'STARY GROMADZYN': {'lat': 53.4000, 'lon': 21.9167},
    'KARWOWO': {'lat': 53.2667, 'lon': 22.3833},
    'OBRYTKI': {'lat':	53.3687798,'lon':22.2742335},
    'ZABAWKA': {'lat' :53.1995772, 'lon': 22.1571153},
    'SIEBURCZYN': 	{ 'lat' : 53.2414279, 'lon' : 22.4342496},
    'SZÓSTAKI' : { 'lat' :53.2833309, 'lon' : 22.4569495},
    'GÓRKI-SYPNIEWO' : { 'lat' : 53.273853, 'lon' : 22.1353291},
    'ŚNIADOWO': 	{'lat':53.0386233, 'lon':21.9902764},
    'SIESTRZANKI': { 'lat':53.306823, 'lon': 22.4139482},
    'BUDY MIKOłAJKA': { 'lat': 53.255353, 'lon' : 22.1711752},
    'MODZELE SKUDOSZE': { 'lat' : 53.0550814, 'lon': 22.1775928},
    'PIĄTNICA WŁOŚCIAŃSKA': { 'lat' :53.1859359, 'lon': 22.1050388},
    'RZADKOWO': {'lat':53.2219677, 'lon':22.143627},
    'PIĄTNICA' : {'lat':53.21542775, 'lon':22.177244553669112},
    'ZOSIN' : {'lat': 53.141755, 'lon': 22.1116481},
    'MORGOWNIKI': {'lat': 53.2275, 'lon': 21.8903},       
    'JANÓW': {'lat': 53.1558, 'lon': 22.1353},            
    'JĘCZNIKI': {'lat': 53.5358, 'lon': 20.9414},         
    'WYK': {'lat': 53.2758, 'lon': 21.9058},              
    'PTAKI': {'lat': 53.3058, 'lon': 21.8481},           
    'ZBÓJNA': {'lat': 53.2358, 'lon': 21.8153},          
    'BUDNE': {'lat': 53.3342, 'lon': 22.4286},            
    'NOWY KRZEW': {'lat': 53.1158, 'lon': 22.2542},       
    'KOZIOŁ': {'lat': 53.4258, 'lon': 21.8986},          
    'MIASTKOWO': {'lat': 53.1508, 'lon': 21.8386},        
    'DĘBOWO': {'lat': 53.6058, 'lon': 22.9253},           
    'BUDY KRANOCKIE': {'lat': 53.1958, 'lon': 22.1486},  
    'STARA ŁOMŻA': {'lat': 53.1658, 'lon': 22.0986},  
    'STAREJ ŁOMŻY': {'lat': 53.1658, 'lon': 22.0986},
    'SIEMIĘ NADRZECZNE': {'lat': 53.1858, 'lon': 22.0253}, 
    'WYRZYKI': {'lat': 53.1058, 'lon': 22.1553},
    'ROGIENICE WIELKIE': {'lat':53.1616, 'lon':22.044},
    'KOBYLIN': {'lat':53.293171789627955, 'lon': 22.14513394107225,}
}

# HELPER FUNCTIONS

In [8]:
def download_db():
    now = datetime.now().strftime("%Y-%m-%d")
    folder_name = "db"
    
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
    
    local_filename = f"{now}_olx.db"
    local_full_path = os.path.join(folder_name, local_filename)

    print(f"Connecting {creds['host']}...")
    
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    
    try:
        ssh.connect(
            hostname=creds["host"],
            port=creds["port"],
            username=creds["user"],
            password=creds["password"]
        )
        
        with SCPClient(ssh.get_transport()) as scp:
            print(f"Pobieranie {creds['remote_path']} -> {local_full_path}")
            scp.get(creds["remote_path"], local_full_path)
            
        print(f"Database saved as: {local_full_path}")
        return local_full_path  
        
    except Exception as e:
        print(f"Error: {e}")
        return None
    finally:
        ssh.close()

In [9]:
def get_latest_db_path(folder="db"):
    files = glob.glob(os.path.join(folder, "*.db"))
    
    if not files:
        print("Not found file with extension 'db'!")
        return None
    
    latest_file = max(files)
    print(f"newest file: {latest_file}")
    return latest_file

In [10]:
def load_data_to_df(db_name: str, table_name: str) -> pl.DataFrame | None:
    conn = None
    try:
        conn = sqlite3.connect(db_name)
        print(f"DB connected: {db_name}")

        query = f"SELECT * FROM {table_name}"
        df = pl.read_database(query, conn)

        print(f"DF loaded: {table_name}")
        print(f"DF shape: {df.shape}")
        return df

    except sqlite3.Error as e:
        print(f"SQLite error: {e}")
        return None
    except Exception as e:
        print(f"Error: {e}")
        return None
    finally:
        if conn:
            conn.close()
            print("DB connection closed.")

In [11]:
def simplify_text(text: str) -> str:
    trans = str.maketrans("ĄĆĘŁŃÓŚŹŻ", "ACELNOSZZ")
    return text.upper().translate(trans)

In [135]:
def add_haversine_distance(df, target_lat, target_lon):
    R = 6371.0
    lat1 = target_lat * np.pi / 180
    lon1 = target_lon * np.pi / 180

    return df.with_columns(
        (
            2 * R * (
                (
                    ((pl.col("LAT") * np.pi / 180 - lat1) / 2).sin()**2 +
                    pl.lit(np.cos(lat1)) * (pl.col("LAT") * np.pi / 180).cos() * ((pl.col("LON") * np.pi / 180 - lon1) / 2).sin()**2
                ).sqrt().arcsin()
            )
        ).cast(pl.Float64).alias("MAIN_CITY_DIST") 
    )

# DOWNLOADING DATA

In [12]:
download_db()

Connecting jan154.mikrus.xyz...
Pobieranie /root/Estate_scrapper/olx.db -> db\2026-03-15_olx.db
Database saved as: db\2026-03-15_olx.db


'db\\2026-03-15_olx.db'

In [13]:
DB_NAME = get_latest_db_path()

newest file: db\2026-03-15_olx.db


In [14]:
data = {}

In [15]:
for TABLE in TABLES:
    data[TABLE] = load_data_to_df(DB_NAME, TABLE)

DB connected: db\2026-03-15_olx.db
DF loaded: offers
DF shape: (320, 12)
DB connection closed.
DB connected: db\2026-03-15_olx.db
DF loaded: offers_tytan
DF shape: (47, 9)
DB connection closed.
DB connected: db\2026-03-15_olx.db
DF loaded: offers_gralczyk
DF shape: (32, 26)
DB connection closed.
DB connected: db\2026-03-15_olx.db
DF loaded: offers_polnoc
DF shape: (108, 30)
DB connection closed.
DB connected: db\2026-03-15_olx.db
DF loaded: analyzed_offers
DF shape: (40135, 12)
DB connection closed.


In [16]:
data.keys()

dict_keys(['offers', 'offers_tytan', 'offers_gralczyk', 'offers_polnoc', 'analyzed_offers'])

# Data processing

## OLX ("offers")

In [17]:
data["offers"].head()

ID,TEXT,LINK,VALUE,CURRENCY,NEGOTIABLE,AREA_M2,PRICE_M2,CITY,DATE_ORIG,LAST_UPDATED,DATE_ISO
str,str,str,i64,str,i64,str,str,str,str,str,str
"""Łomża_2134_99000""","""Działka na sprzedaż z dostępem do asfaltu i lasu99 000 złŁomża - 10 listopada 20252 134 m² - 46.39 zł/m²Obserwuj""","""https://www.otodom.pl/pl/oferta/dzialka-na-sprzedaz-z-dostepem-do-asfaltu-i-lasu-ID4yVmd.html""",99000,"""zł""",0,"""2134""","""46.39""","""Łomża""","""Łomża - 10 listopada 2025""","""2025-11-11""","""2025-11-10"""
"""Zawady_1000_130000""","""Sprzedam działkę. Elżbiecin.130 000 złZawady - 09 listopada 20251 000 m² - 130 zł/m²Obserwuj""","""/d/oferta/sprzedam-dzialke-elzbiecin-CID3-ID11ZDLY.html""",130000,"""zł""",0,"""1000""","""130.0""","""Zawady""","""Zawady - 11 lutego 2026""","""2025-11-11""","""2026-02-11"""
"""Elżbiecin_12200_1490000""","""Działka na sprzedaż1 490 000 złElżbiecin - 09 listopada 202512 200 m² - 122.13 zł/m²Obserwuj""","""/d/oferta/dzialka-na-sprzedaz-CID3-ID11ZvRU.html""",1490000,"""zł""",0,"""12200""","""122.13""","""Elżbiecin""","""Elżbiecin - 10 lutego 2026""","""2025-11-11""","""2026-02-10"""
"""Jurki_3000_67500""","""Sprzedam piękną dużą 1200m2 działkę przy rzece Pisie, w Jurkach.67 500 złJurki - 06 listopada 20253 000 m² - 22.5 zł/m²Obserwuj""","""/d/oferta/sprzedam-piekna-duza-1200m2-dzialke-przy-rzece-pisie-w-jurkach-CID3-IDH8Ttm.html""",67500,"""zł""",0,"""3000""","""22.5""","""Jurki""","""Jurki - 06 listopada 2025""","""2025-11-11""","""2025-11-06"""
"""Łomża_1000_129500""","""Sprzedam działki pod zabudowę mieszkaniową jednorodzinną w Giełczynie129 500 złŁomża - 06 listopada 20251 000 m² - 129.5 zł/m²Obserwuj""","""/d/oferta/sprzedam-dzialki-pod-zabudowe-mieszkaniowa-jednorodzinna-w-gielczynie-CID3-IDZoZkx.html""",129500,"""zł""",0,"""1000""","""129.5""","""Łomża""","""Łomża - 06 listopada 2025""","""2025-11-11""","""2025-11-06"""


In [18]:
data["offers"] = data["offers"].rename({"DATE_ISO":"DATE_ADDED"})

In [19]:
data["offers"] = data["offers"].with_columns(
    pl.col(["LAST_UPDATED", "DATE_ADDED"]).str.to_date(format="%Y-%m-%d")
            )

In [20]:
data["offers"] = data["offers"].with_columns(
    pl.col(["AREA_M2", "PRICE_M2"]).cast(pl.Float64)
)

In [21]:
data["offers"] = data["offers"].with_columns(
    pl.col("CITY").str.to_uppercase()
)

In [22]:
geo_df = pl.DataFrame([
    {"CITY": k, "LAT": v["lat"], "LON": v["lon"]} 
    for k, v in CITIES_GEO.items()
])

data["offers"] = data["offers"].join(geo_df, on="CITY", how="left")

In [23]:
print(data["offers"].null_count())

shape: (1, 14)
┌─────┬──────┬──────┬───────┬───┬──────────────┬────────────┬─────┬─────┐
│ ID  ┆ TEXT ┆ LINK ┆ VALUE ┆ … ┆ LAST_UPDATED ┆ DATE_ADDED ┆ LAT ┆ LON │
│ --- ┆ ---  ┆ ---  ┆ ---   ┆   ┆ ---          ┆ ---        ┆ --- ┆ --- │
│ u32 ┆ u32  ┆ u32  ┆ u32   ┆   ┆ u32          ┆ u32        ┆ u32 ┆ u32 │
╞═════╪══════╪══════╪═══════╪═══╪══════════════╪════════════╪═════╪═════╡
│ 0   ┆ 0    ┆ 0    ┆ 0     ┆ … ┆ 0            ┆ 3          ┆ 0   ┆ 0   │
└─────┴──────┴──────┴───────┴───┴──────────────┴────────────┴─────┴─────┘


In [24]:
data["offers"] = data["offers"].with_columns(pl.col("DATE_ADDED").fill_null(pl.col("LAST_UPDATED"))
)

In [25]:
data["offers"] = data["offers"].with_columns(
    pl.col("DATE_ADDED").dt.strftime("%A").alias("DAY_NAME_EN")
)

In [26]:
data["offers"].head(5)

ID,TEXT,LINK,VALUE,CURRENCY,NEGOTIABLE,AREA_M2,PRICE_M2,CITY,DATE_ORIG,LAST_UPDATED,DATE_ADDED,LAT,LON,DAY_NAME_EN
str,str,str,i64,str,i64,f64,f64,str,str,date,date,f64,f64,str
"""Łomża_2134_99000""","""Działka na sprzedaż z dostępem do asfaltu i lasu99 000 złŁomża - 10 listopada 20252 134 m² - 46.39 zł/m²Obserwuj""","""https://www.otodom.pl/pl/oferta/dzialka-na-sprzedaz-z-dostepem-do-asfaltu-i-lasu-ID4yVmd.html""",99000,"""zł""",0,2134.0,46.39,"""ŁOMŻA""","""Łomża - 10 listopada 2025""",2025-11-11,2025-11-10,53.1781,22.0592,"""Monday"""
"""Zawady_1000_130000""","""Sprzedam działkę. Elżbiecin.130 000 złZawady - 09 listopada 20251 000 m² - 130 zł/m²Obserwuj""","""/d/oferta/sprzedam-dzialke-elzbiecin-CID3-ID11ZDLY.html""",130000,"""zł""",0,1000.0,130.0,"""ZAWADY""","""Zawady - 11 lutego 2026""",2025-11-11,2026-02-11,53.1539,22.6653,"""Wednesday"""
"""Elżbiecin_12200_1490000""","""Działka na sprzedaż1 490 000 złElżbiecin - 09 listopada 202512 200 m² - 122.13 zł/m²Obserwuj""","""/d/oferta/dzialka-na-sprzedaz-CID3-ID11ZvRU.html""",1490000,"""zł""",0,12200.0,122.13,"""ELŻBIECIN""","""Elżbiecin - 10 lutego 2026""",2025-11-11,2026-02-10,53.1672,22.1231,"""Tuesday"""
"""Jurki_3000_67500""","""Sprzedam piękną dużą 1200m2 działkę przy rzece Pisie, w Jurkach.67 500 złJurki - 06 listopada 20253 000 m² - 22.5 zł/m²Obserwuj""","""/d/oferta/sprzedam-piekna-duza-1200m2-dzialke-przy-rzece-pisie-w-jurkach-CID3-IDH8Ttm.html""",67500,"""zł""",0,3000.0,22.5,"""JURKI""","""Jurki - 06 listopada 2025""",2025-11-11,2025-11-06,53.3983,22.1114,"""Thursday"""
"""Łomża_1000_129500""","""Sprzedam działki pod zabudowę mieszkaniową jednorodzinną w Giełczynie129 500 złŁomża - 06 listopada 20251 000 m² - 129.5 zł/m²Obserwuj""","""/d/oferta/sprzedam-dzialki-pod-zabudowe-mieszkaniowa-jednorodzinna-w-gielczynie-CID3-IDZoZkx.html""",129500,"""zł""",0,1000.0,129.5,"""ŁOMŻA""","""Łomża - 06 listopada 2025""",2025-11-11,2025-11-06,53.1781,22.0592,"""Thursday"""


In [27]:
data["offers"].null_count()

ID,TEXT,LINK,VALUE,CURRENCY,NEGOTIABLE,AREA_M2,PRICE_M2,CITY,DATE_ORIG,LAST_UPDATED,DATE_ADDED,LAT,LON,DAY_NAME_EN
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [28]:
data["offers"].filter(pl.col("LAT").is_null())["CITY"].unique().to_list()

[]

## TYTAN

In [29]:
data["offers_tytan"].head(2)

ID,LOCATION,NAME,AREA_M2,PRICE,PRICE_M2,LINK,LAST_UPDATED,DATE_ADDED
str,str,str,i64,i64,f64,str,str,str
"""Szóstaki_18800_75000""","""Szóstaki""","""Działka rolna na terenie Parku Narodowego.""",18800,75000,4.0,"""https://tytan-nieruchomosci.com/nieruchomosci/?f_sectionName1=Lot&f_sectionName2=Sale&f_location_locality[0]=&mapa=&mapaX1=&mapaY1=&mapaX2=&mapaY2=&mapaCX=&mapaCY=&mapaCZ=0&mapaVis=0&submit=Szukaj&f_price_amountMin=&f_price_amountMax=&f_totalAreaMin=&f_totalAreaMax=&offset_22934=0&cid=22934/dzialka-sprzedaz-188m2-75000pln-szostaki-podlaskie,1577/18682/OGS""","""2026-03-14""","""2025-11-16"""
"""Szóstaki_7700_58000""","""Szóstaki""","""Działka rolna Szostaki gmina Jedwabne.""",7700,58000,7.0,"""https://tytan-nieruchomosci.com/nieruchomosci/?f_sectionName1=Lot&f_sectionName2=Sale&f_location_locality[0]=&mapa=&mapaX1=&mapaY1=&mapaX2=&mapaY2=&mapaCX=&mapaCY=&mapaCZ=0&mapaVis=0&submit=Szukaj&f_price_amountMin=&f_price_amountMax=&f_totalAreaMin=&f_totalAreaMax=&offset_22934=0&cid=22934/dzialka-sprzedaz-077m2-58000pln-szostaki-podlaskie,1576/18682/OGS""","""2026-03-14""","""2025-11-16"""


In [30]:
data["offers_tytan"] = data["offers_tytan"].with_columns(
    pl.col(["LAST_UPDATED", "DATE_ADDED"]).str.to_date(format="%Y-%m-%d")
            )

In [31]:
data["offers_tytan"] = data["offers_tytan"].rename({"LOCATION":"CITY"})

In [32]:
data["offers_tytan"] = data["offers_tytan"].with_columns(
    pl.col("CITY").str.to_uppercase()
)

In [33]:
data["offers_tytan"] = data["offers_tytan"].with_columns(
    pl.col("DATE_ADDED").dt.strftime("%A").alias("DAY_NAME_EN")
)

In [34]:
geo_df = pl.DataFrame([
    {"CITY": k, "LAT": v["lat"], "LON": v["lon"]} 
    for k, v in CITIES_GEO.items()
])

data["offers_tytan"] = data["offers_tytan"].join(geo_df, on="CITY", how="left")

In [35]:
data["offers_tytan"].filter(pl.col("LAT").is_null())["CITY"].unique().to_list()

[]

# GRALCZYK

In [36]:
data["offers_gralczyk"] = data["offers_gralczyk"].rename({"FIRST_ADDED":"DATE_ADDED"})

In [37]:
data["offers_gralczyk"] = data["offers_gralczyk"].with_columns(
    pl.col(["LAST_UPDATED", "DATE_ADDED"]).str.to_date(format="%Y-%m-%d")
            )

In [38]:
data["offers_gralczyk"] = data["offers_gralczyk"].rename({"Cena":"PRICE"})
data["offers_gralczyk"] = data["offers_gralczyk"].rename({"Cena za m2":"PRICE_M2"})

In [39]:
data["offers_gralczyk"] = data["offers_gralczyk"].rename({"Powierzchnia":"AREA_M2"})

In [40]:
data["offers_gralczyk"] = data["offers_gralczyk"].with_columns(
    pl.col("AREA_M2")
    .str.replace("  m2", "")
    .str.replace(" ", "")
    .str.replace(",", ".")       
    .cast(pl.Float64)
)

In [41]:
data["offers_gralczyk"]

ID,DATE_ADDED,PRICE,PRICE_M2,Długość,Forma własności,Gaz,Kanalizacja,LAST_UPDATED,LINK,Liczba linii tel.,Numer oferty,AREA_M2,Powierzchnia zabudowy,Przeznaczenie,Prąd,Szerokość,Sąsiedztwo,Typ drogi,Typ działki,Typ ogrodzenia,Typ transakcji,Woda,Wygląd działki,Wysokość min.,Łącze internetowe
str,date,str,str,str,str,str,str,date,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str
"""lomza-nowogrodzka-dzialka-inwestycyjna-z-ogromnym-potencjalem""",2025-11-22,"""1650000""","""201.88""","""155,5""","""własność""","""miejski""","""miejska""",2026-03-14,"""https://www.goralczyknieruchomosci.pl/oferta/2485-3-1/lomza-nowogrodzka-dzialka-inwestycyjna-z-ogromnym-potencjalem""",null,"""2485""",8173.0,"""40""","""przemysłowo-usługowa""","""jest""","""52,5""","""zabudowa jednorodzinna oraz komercyjna""","""asfaltowa""","""inwestycyjna""","""brak""","""Sprzedaż""","""miejska""","""teren płaski, pusta""","""10""","""światłowód"""
"""lomza-bohdana-winiarskiego-dzialka-pod-budowe-twojego-wymarzonego-domu""",2025-11-22,"""265000""","""350.99""","""45""","""własność""","""miejski""","""miejska""",2026-03-14,"""https://www.goralczyknieruchomosci.pl/oferta/2307-3-1/lomza-bohdana-winiarskiego-dzialka-pod-budowe-twojego-wymarzonego-domu""",null,"""2307""",755.0,null,"""budownictwo jednorodzinne""","""jest""","""16""","""Zabudowa jednorodzinna""","""szutrowa""","""budowlana""","""brak""","""Sprzedaż""","""miejska""","""teren płaski, pusta""",null,"""brak"""
"""lomza-kalinowa-wyjatkowa-dzialka-budowlana-w-lomzy-ul-kalinowa-os-piaski""",2025-11-22,"""465000""","""178.98""","""77,5""","""własność""","""miejski""","""miejska""",2026-03-14,"""https://www.goralczyknieruchomosci.pl/oferta/2429-3-1/lomza-kalinowa-wyjatkowa-dzialka-budowlana-w-lomzy-ul-kalinowa-os-piaski""",null,"""2429""",2598.0,"""13""","""budownictwo jednorodzinne""","""jest""","""33,5""","""Zabudowa jednorodzinna""","""asfaltowa""","""budowlana""","""brak""","""Sprzedaż""","""miejska""","""teren płaski, pusta, linia brzegowa""","""10""","""światłowód"""
"""lomza-laczna-dzialka-budowlana-w-bardzo-atrakcyjnej-lokalizacji""",2025-11-22,"""165000""","""471.43""","""22,5""","""własność""","""miejski""","""miejska""",2026-03-02,"""https://www.goralczyknieruchomosci.pl/oferta/2114-3-1/lomza-laczna-dzialka-budowlana-w-bardzo-atrakcyjnej-lokalizacji""",null,"""2114""",350.0,"""35""","""budownictwo jednorodzinne""","""jest""","""15,5""","""Zabudowa jednorodzinna""","""kostka""","""budowlana""","""metalowe""","""Sprzedaż""","""miejska""","""zabudowana""","""2""","""światłowód"""
"""lomza-krolowej-bony-dzialka-budowlana-w-swietnej-lokalizacji""",2025-11-22,"""299000""","""327.49""","""41""","""własność""","""w drodze""","""miejska""",2026-03-02,"""https://www.goralczyknieruchomosci.pl/oferta/2279-3-1/lomza-krolowej-bony-dzialka-budowlana-w-swietnej-lokalizacji""","""1""","""2279""",913.0,"""30""","""budownictwo jednorodzinne""","""jest""","""24,5""","""Zabudowa jednorodzinna""","""szutrowa""","""budowlana""","""brak""","""Sprzedaż""","""miejska""","""teren zróżnicowany, pusta""","""2""","""brak"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""szczepankowo-dzialki-w-atrakcyjnych-cenach-blisko-natury""",2026-03-11,"""80343""","""79""","""49""","""własność""","""brak""","""szambo""",2026-03-14,"""https://www.goralczyknieruchomosci.pl/oferta/2133-3-1/szczepankowo-dzialki-w-atrakcyjnych-cenach-blisko-natury""",null,"""2133""",1017.0,"""20""","""budownictwo jednorodzinne""","""brak""","""22""","""zabudowa jednorodzinna, las, tereny zielone""","""asfaltowa""","""budowlana""","""brak""","""Sprzedaż""","""studnia""","""pusta""","""6""","""brak"""
"""motyka-malownicze-dzialki-w-sasiedztwie-lasu-motyka""",2026-03-12,"""90000""","""79.51""",null,"""własność""","""brak""","""brak""",2026-03-14,"""https://www.goralczyknieruchomosci.pl/oferta/2258-3-1/motyka-malownicze-dzialki-w-sasiedztwie-lasu-motyka""","""1""","""2258""",1132.0,null,"""budownictwo jednorodzinne""","""do 100m""",null,null

In [42]:
data["offers_gralczyk"] = data["offers_gralczyk"].with_columns(
    pl.col("ID")
    .str.replace_all("dzialka", "")
    .str.replace_all("-", " ")      
    .str.strip_chars()              
    .str.to_uppercase()             
    .alias("CITY")                  
)

In [43]:
data["offers_gralczyk"]

ID,DATE_ADDED,PRICE,PRICE_M2,Długość,Forma własności,Gaz,Kanalizacja,LAST_UPDATED,LINK,Liczba linii tel.,Numer oferty,AREA_M2,Powierzchnia zabudowy,Przeznaczenie,Prąd,Szerokość,Sąsiedztwo,Typ drogi,Typ działki,Typ ogrodzenia,Typ transakcji,Woda,Wygląd działki,Wysokość min.,Łącze internetowe,CITY
str,date,str,str,str,str,str,str,date,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""lomza-nowogrodzka-dzialka-inwestycyjna-z-ogromnym-potencjalem""",2025-11-22,"""1650000""","""201.88""","""155,5""","""własność""","""miejski""","""miejska""",2026-03-14,"""https://www.goralczyknieruchomosci.pl/oferta/2485-3-1/lomza-nowogrodzka-dzialka-inwestycyjna-z-ogromnym-potencjalem""",null,"""2485""",8173.0,"""40""","""przemysłowo-usługowa""","""jest""","""52,5""","""zabudowa jednorodzinna oraz komercyjna""","""asfaltowa""","""inwestycyjna""","""brak""","""Sprzedaż""","""miejska""","""teren płaski, pusta""","""10""","""światłowód""","""LOMZA NOWOGRODZKA INWESTYCYJNA Z OGROMNYM POTENCJALEM"""
"""lomza-bohdana-winiarskiego-dzialka-pod-budowe-twojego-wymarzonego-domu""",2025-11-22,"""265000""","""350.99""","""45""","""własność""","""miejski""","""miejska""",2026-03-14,"""https://www.goralczyknieruchomosci.pl/oferta/2307-3-1/lomza-bohdana-winiarskiego-dzialka-pod-budowe-twojego-wymarzonego-domu""",null,"""2307""",755.0,null,"""budownictwo jednorodzinne""","""jest""","""16""","""Zabudowa jednorodzinna""","""szutrowa""","""budowlana""","""brak""","""Sprzedaż""","""miejska""","""teren płaski, pusta""",null,"""brak""","""LOMZA BOHDANA WINIARSKIEGO POD BUDOWE TWOJEGO WYMARZONEGO DOMU"""
"""lomza-kalinowa-wyjatkowa-dzialka-budowlana-w-lomzy-ul-kalinowa-os-piaski""",2025-11-22,"""465000""","""178.98""","""77,5""","""własność""","""miejski""","""miejska""",2026-03-14,"""https://www.goralczyknieruchomosci.pl/oferta/2429-3-1/lomza-kalinowa-wyjatkowa-dzialka-budowlana-w-lomzy-ul-kalinowa-os-piaski""",null,"""2429""",2598.0,"""13""","""budownictwo jednorodzinne""","""jest""","""33,5""","""Zabudowa jednorodzinna""","""asfaltowa""","""budowlana""","""brak""","""Sprzedaż""","""miejska""","""teren płaski, pusta, linia brzegowa""","""10""","""światłowód""","""LOMZA KALINOWA WYJATKOWA BUDOWLANA W LOMZY UL KALINOWA OS PIASKI"""
"""lomza-laczna-dzialka-budowlana-w-bardzo-atrakcyjnej-lokalizacji""",2025-11-22,"""165000""","""471.43""","""22,5""","""własność""","""miejski""","""miejska""",2026-03-02,"""https://www.goralczyknieruchomosci.pl/oferta/2114-3-1/lomza-laczna-dzialka-budowlana-w-bardzo-atrakcyjnej-lokalizacji""",null,"""2114""",350.0,"""35""","""budownictwo jednorodzinne""","""jest""","""15,5""","""Zabudowa jednorodzinna""","""kostka""","""budowlana""","""metalowe""","""Sprzedaż""","""miejska""","""zabudowana""","""2""","""światłowód""","""LOMZA LACZNA BUDOWLANA W BARDZO ATRAKCYJNEJ LOKALIZACJI"""
"""lomza-krolowej-bony-dzialka-budowlana-w-swietnej-lokalizacji""",2025-11-22,"""299000""","""327.49""","""41""","""własność""","""w drodze""","""miejska""",2026-03-02,"""https://www.goralczyknieruchomosci.pl/oferta/2279-3-1/lomza-krolowej-bony-dzialka-budowlana-w-swietnej-lokalizacji""","""1""","""2279""",913.0,"""30""","""budownictwo jednorodzinne""","""jest""","""24,5""","""Zabudowa jednorodzinna""","""szutrowa""","""budowlana""","""brak""","""Sprzedaż""","""miejska""","""teren zróżnicowany, pusta""","""2""","""brak""","""LOMZA KROLOWEJ BONY BUDOWLANA W SWIETNEJ LOKALIZACJI"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""szczepankowo-dzialki-w-atrakcyjnych-cenach-blisko-natury""",2026-03-11,"""80343""","""79""","""49""","""własność""","""brak""","""szambo""",2026-03-14,"""https://www.goralczyknieruchomosci.pl/oferta/2133-3-1/szczepankowo-dzialki-w-atrakcyjnych-cenach-blisko-natury""",null,"""2133""",1017.0,"""20""","""budownictwo jednorodzinne""","""brak""","""22""","""zabudowa jednorodzinna, las, tereny zielone""","""asfaltowa""","""budowlana""","""brak""","""Sprzedaż""","""studnia""","""pusta""","""6""","""brak""","""SZCZEPANK

In [44]:
data["offers_gralczyk"] = data["offers_gralczyk"].with_columns(
    pl.col("CITY")
    .str.to_uppercase()
    .str.replace_all("Ł", "L")
    .str.replace_all("Ó", "O")
    .str.replace_all("Ś", "S")
    .str.replace_all("Ż", "Z")
    .str.replace_all("Ź", "Z")
    .str.replace_all("Ć", "C")
    .str.replace_all("Ń", "N")
    .str.replace_all("Ą", "A")
    .str.replace_all("Ę", "E")
    .alias("SEARCH_CITY")
)

condition = pl.when(False).then(None)

for original_city in CITIES_GEO.keys():
    simple_city = simplify_text(original_city) 
    
    condition = condition.when(
        pl.col("SEARCH_CITY").str.contains(simple_city)
    ).then(pl.lit(original_city)) 

data["offers_gralczyk"] = data["offers_gralczyk"].with_columns(
    condition.alias("CLEAN_CITY")
)

data["offers_gralczyk"] = data["offers_gralczyk"].drop("SEARCH_CITY")

In [45]:
data["offers_gralczyk"].filter(pl.col("CLEAN_CITY").is_null())["CITY"].unique().to_list()

[]

In [46]:
data["offers_gralczyk"] = data["offers_gralczyk"].drop( ["CITY"])
data["offers_gralczyk"] = data["offers_gralczyk"].rename({"CLEAN_CITY":"CITY"})

In [47]:
data["offers_gralczyk"] = data["offers_gralczyk"].join(geo_df, on="CITY", how="left")

In [48]:
data["offers_gralczyk"].head(2)

ID,DATE_ADDED,PRICE,PRICE_M2,Długość,Forma własności,Gaz,Kanalizacja,LAST_UPDATED,LINK,Liczba linii tel.,Numer oferty,AREA_M2,Powierzchnia zabudowy,Przeznaczenie,Prąd,Szerokość,Sąsiedztwo,Typ drogi,Typ działki,Typ ogrodzenia,Typ transakcji,Woda,Wygląd działki,Wysokość min.,Łącze internetowe,CITY,LAT,LON
str,date,str,str,str,str,str,str,date,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64
"""lomza-nowogrodzka-dzialka-inwestycyjna-z-ogromnym-potencjalem""",2025-11-22,"""1650000""","""201.88""","""155,5""","""własność""","""miejski""","""miejska""",2026-03-14,"""https://www.goralczyknieruchomosci.pl/oferta/2485-3-1/lomza-nowogrodzka-dzialka-inwestycyjna-z-ogromnym-potencjalem""",null,"""2485""",8173.0,"""40""","""przemysłowo-usługowa""","""jest""","""52,5""","""zabudowa jednorodzinna oraz komercyjna""","""asfaltowa""","""inwestycyjna""","""brak""","""Sprzedaż""","""miejska""","""teren płaski, pusta""","""10""","""światłowód""","""NOWOGRÓD""",53.2264,21.8806
"""lomza-bohdana-winiarskiego-dzialka-pod-budowe-twojego-wymarzonego-domu""",2025-11-22,"""265000""","""350.99""","""45""","""własność""","""miejski""","""miejska""",2026-03-14,"""https://www.goralczyknieruchomosci.pl/oferta/2307-3-1/lomza-bohdana-winiarskiego-dzialka-pod-budowe-twojego-wymarzonego-domu""",null,"""2307""",755.0,null,"""budownictwo jednorodzinne""","""jest""","""16""","""Zabudowa jednorodzinna""","""szutrowa""","""budowlana""","""brak""","""Sprzedaż""","""miejska""","""teren płaski, pusta""",null,"""brak""","""ŁOMŻA""",53.1781,22.0592


# POLNOC

In [49]:
data["offers_polnoc"] = data["offers_polnoc"].rename({"FIRST_ADDED":"DATE_ADDED"})

In [50]:
data["offers_polnoc"] = data["offers_polnoc"].with_columns(
    pl.col(["LAST_UPDATED", "DATE_ADDED"]).str.to_date(format="%Y-%m-%d")
            )

In [51]:
data["offers_polnoc"] = data["offers_polnoc"].rename({"Cena":"PRICE"})
data["offers_polnoc"] = data["offers_polnoc"].rename({"Cena za m2":"PRICE_M2"})
data["offers_polnoc"] = data["offers_polnoc"].rename({"Powierzchnia działki":"AREA_M2"})

In [52]:
data["offers_polnoc"] = data["offers_polnoc"].filter(
    (pl.col("AREA_M2").is_not_null()) & 
    (pl.col("ID").str.contains("dzialka"))
)

In [53]:
data["offers_polnoc"]

ID,DATE_ADDED,LAST_UPDATED,PRICE,PRICE_M2,Ciepła woda,Długość działki (mb.),Garaż/Miejsca parkingowe,Id oferty,Kształt działki,LINK,Liczba pięter,Liczba pokoi,Ogrodzenie,Piętro,Plan miejscowy,Powierzchnia,AREA_M2,Powierzchnia użytkowa,Przeznaczenie działki,Rodzaj budynku,Rok budowy,Rynek,Stan budynku,Szerokość działki (mb.),Typ domu,Typ kuchni,Tytuł,Ukształtowanie działki,Winda
str,date,date,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""dzialka-pupki""",2025-11-22,2026-01-31,"""94000.0""","""50.0""",null,"""70,00 m""",null,"""46804/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-pupki/""",null,null,"""Brak""",null,"""Brak planu miejscowego""",null,"""1880.0""",null,"""Budowlana""",null,null,null,null,"""28,00 m""",null,null,"""Na sprzedaż działki budowlane w Pupkach, gm.Turośl""","""Płaska""",null
"""dzialka-stare-kupiski-ul-wspolna""",2025-11-22,2025-12-02,"""129000.0""","""162.88""",null,"""29,00 m""",null,"""46747/3877/OGS""","""Kwadrat""","""https://polnoc.pl/oferty/dzialka-stare-kupiski-ul-wspolna/""",null,null,"""Brak""",null,"""Brak planu miejscowego""",null,"""792.0""",null,"""Budowlana""",null,null,null,null,"""28,00 m""",null,null,"""Działka budowlana z WZ w Starych Kupiskach!""","""Płaska""",null
"""dzialka-lomza-zdrojowa""",2025-11-22,2026-01-31,"""249000.0""","""214.47""",null,null,null,"""46675/3877/OGS""","""Trapez""","""https://polnoc.pl/oferty/dzialka-lomza-zdrojowa/""",null,null,"""Mieszane""",null,"""Brak planu miejscowego""",null,"""1161.0""",null,"""Budowlana""",null,null,null,null,null,null,null,"""Działka w Łomży na ul. Zdrojowej!""","""Zróżnicowana""",null
"""dzialka-lomza-waska""",2025-11-22,2026-01-31,"""205000.0""","""272.97""",null,"""26,00 m""",null,"""44481/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-lomza-waska/""",null,null,"""Brak""",null,"""Brak planu miejscowego""",null,"""751.0""",null,"""Budowlana""",null,null,null,null,"""29,00 m""",null,null,"""Malownicza działka w Łomży na ul. Wąskiej!""","""Płaska""",null
"""dzialka-lomza-grabowa""",2025-11-22,2026-01-31,"""239000.0""","""247.02""",null,"""32,00 m""",null,"""46667/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-lomza-grabowa/""",null,null,"""Brak""",null,"""Brak planu miejscowego""",null,"""1008.0""",null,"""Budowlana""",null,null,null,null,"""33,50 m""",null,null,"""Przestronne działki w Łomży na ul. Grabowej!""","""Płaska""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""dzialka-lomza-ul-dzialkowa""",2025-12-31,2026-02-14,"""370000.0""","""430.23""",null,null,null,"""46948/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-lomza-ul-dzialkowa/""",null,null,"""Brak""",null,"""Plan zagospodarowania przestrzennego""",null,"""860.0""",null,"""Budowlana""",null,null,null,null,null,null,null,"""Działka – Łomża, ul. Działkowa""","""Płaska""",null
"""dzialka-lomza-ul-dzialkowa-2""",2026-01-09,2026-01-31,"""370000.0""","""430.23""",null,null,null,"""46948/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-lomza-ul-dzialkowa-2/""",null,null,"""Brak""",null,"""Plan zagospodarowania przestrzennego""",null,"""860.0""",null,"""Budowlana""",null,null,null,null,null,null,null,"""Działka – Łomża, ul. Działkowa""","""Płaska""",null
"""dzialka-budy-czarnockie-2""",2026-01-09,2026-01-31,"""105000.0""","""106.49""",null,"""44,50 m""",null,"""47008/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-budy-czarnockie-2/""",null,null,"""Brak""",null,"""Brak planu miejscowego""",null,"""986.0""",null,"""Budowlana""",null,null,null,null,"""22,00 m""",null,null,"""Atrakcyjna działka w Budach Czarnockich!""","""Płaska""",null


In [54]:
data["offers_polnoc"] = data["offers_polnoc"].with_columns(
    pl.col("ID")
    .str.replace_all("dzialka", "")
    .str.replace_all("-", " ")      
    .str.strip_chars()              
    .str.to_uppercase()             
    .alias("CITY")                  
)

In [55]:
data["offers_polnoc"]

ID,DATE_ADDED,LAST_UPDATED,PRICE,PRICE_M2,Ciepła woda,Długość działki (mb.),Garaż/Miejsca parkingowe,Id oferty,Kształt działki,LINK,Liczba pięter,Liczba pokoi,Ogrodzenie,Piętro,Plan miejscowy,Powierzchnia,AREA_M2,Powierzchnia użytkowa,Przeznaczenie działki,Rodzaj budynku,Rok budowy,Rynek,Stan budynku,Szerokość działki (mb.),Typ domu,Typ kuchni,Tytuł,Ukształtowanie działki,Winda,CITY
str,date,date,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""dzialka-pupki""",2025-11-22,2026-01-31,"""94000.0""","""50.0""",null,"""70,00 m""",null,"""46804/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-pupki/""",null,null,"""Brak""",null,"""Brak planu miejscowego""",null,"""1880.0""",null,"""Budowlana""",null,null,null,null,"""28,00 m""",null,null,"""Na sprzedaż działki budowlane w Pupkach, gm.Turośl""","""Płaska""",null,"""PUPKI"""
"""dzialka-stare-kupiski-ul-wspolna""",2025-11-22,2025-12-02,"""129000.0""","""162.88""",null,"""29,00 m""",null,"""46747/3877/OGS""","""Kwadrat""","""https://polnoc.pl/oferty/dzialka-stare-kupiski-ul-wspolna/""",null,null,"""Brak""",null,"""Brak planu miejscowego""",null,"""792.0""",null,"""Budowlana""",null,null,null,null,"""28,00 m""",null,null,"""Działka budowlana z WZ w Starych Kupiskach!""","""Płaska""",null,"""STARE KUPISKI UL WSPOLNA"""
"""dzialka-lomza-zdrojowa""",2025-11-22,2026-01-31,"""249000.0""","""214.47""",null,null,null,"""46675/3877/OGS""","""Trapez""","""https://polnoc.pl/oferty/dzialka-lomza-zdrojowa/""",null,null,"""Mieszane""",null,"""Brak planu miejscowego""",null,"""1161.0""",null,"""Budowlana""",null,null,null,null,null,null,null,"""Działka w Łomży na ul. Zdrojowej!""","""Zróżnicowana""",null,"""LOMZA ZDROJOWA"""
"""dzialka-lomza-waska""",2025-11-22,2026-01-31,"""205000.0""","""272.97""",null,"""26,00 m""",null,"""44481/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-lomza-waska/""",null,null,"""Brak""",null,"""Brak planu miejscowego""",null,"""751.0""",null,"""Budowlana""",null,null,null,null,"""29,00 m""",null,null,"""Malownicza działka w Łomży na ul. Wąskiej!""","""Płaska""",null,"""LOMZA WASKA"""
"""dzialka-lomza-grabowa""",2025-11-22,2026-01-31,"""239000.0""","""247.02""",null,"""32,00 m""",null,"""46667/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-lomza-grabowa/""",null,null,"""Brak""",null,"""Brak planu miejscowego""",null,"""1008.0""",null,"""Budowlana""",null,null,null,null,"""33,50 m""",null,null,"""Przestronne działki w Łomży na ul. Grabowej!""","""Płaska""",null,"""LOMZA GRABOWA"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""dzialka-lomza-ul-dzialkowa""",2025-12-31,2026-02-14,"""370000.0""","""430.23""",null,null,null,"""46948/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-lomza-ul-dzialkowa/""",null,null,"""Brak""",null,"""Plan zagospodarowania przestrzennego""",null,"""860.0""",null,"""Budowlana""",null,null,null,null,null,null,null,"""Działka – Łomża, ul. Działkowa""","""Płaska""",null,"""LOMZA UL DZIALKOWA"""
"""dzialka-lomza-ul-dzialkowa-2""",2026-01-09,2026-01-31,"""370000.0""","""430.23""",null,null,null,"""46948/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-lomza-ul-dzialkowa-2/""",null,null,"""Brak""",null,"""Plan zagospodarowania przestrzennego""",null,"""860.0""",null,"""Budowlana""",null,null,null,null,null,null,null,"""Działka – Łomża, ul. Działkowa""","""Płaska""",null,"""LOMZA UL DZIALKOWA 2"""
"""dzialka-budy-czarnockie-2""",2026-01-09,2026-01-31,"""105000.0""","""106.49""",null,"""44,50 m""",null,"""47008/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-budy-czarnockie-2/""",null,null,"""Brak""",null,"""Brak planu miejscowego""",null,"""986.0""",null,"""Budowlana""",null,null,null,null,"""22,00 m""",null,null,"""Atrakcyjna działka w Budach Czarnockich!""","""Płaska""",null,"""BUDY CZARNOCKIE 2"""


In [56]:
data["offers_polnoc"] = data["offers_polnoc"].with_columns(
    pl.col(["AREA_M2", "PRICE_M2", "PRICE"]).cast(pl.Float64)
)

In [57]:
data["offers_polnoc"] = data["offers_polnoc"].with_columns(
    pl.col("CITY")
    .str.to_uppercase()
    .str.replace_all("Ł", "L")
    .str.replace_all("Ó", "O")
    .str.replace_all("Ś", "S")
    .str.replace_all("Ż", "Z")
    .str.replace_all("Ź", "Z")
    .str.replace_all("Ć", "C")
    .str.replace_all("Ń", "N")
    .str.replace_all("Ą", "A")
    .str.replace_all("Ę", "E")
    .alias("SEARCH_CITY")
)

condition = pl.when(False).then(None)

for original_city in CITIES_GEO.keys():
    simple_city = simplify_text(original_city) 
    
    condition = condition.when(
        pl.col("SEARCH_CITY").str.contains(simple_city)
    ).then(pl.lit(original_city)) 

data["offers_polnoc"] = data["offers_polnoc"].with_columns(
    condition.alias("CLEAN_CITY")
)

data["offers_polnoc"] = data["offers_polnoc"].drop("SEARCH_CITY")

In [58]:
data["offers_polnoc"].filter(pl.col("CLEAN_CITY").is_null())["CITY"].unique().to_list()

[]

In [59]:
data["offers_polnoc"] = data["offers_polnoc"].drop( ["CITY"])
data["offers_polnoc"] = data["offers_polnoc"].rename({"CLEAN_CITY":"CITY"})
data["offers_polnoc"] = data["offers_polnoc"].join(geo_df, on="CITY", how="left")

In [60]:
data["offers_polnoc"]

ID,DATE_ADDED,LAST_UPDATED,PRICE,PRICE_M2,Ciepła woda,Długość działki (mb.),Garaż/Miejsca parkingowe,Id oferty,Kształt działki,LINK,Liczba pięter,Liczba pokoi,Ogrodzenie,Piętro,Plan miejscowy,Powierzchnia,AREA_M2,Powierzchnia użytkowa,Przeznaczenie działki,Rodzaj budynku,Rok budowy,Rynek,Stan budynku,Szerokość działki (mb.),Typ domu,Typ kuchni,Tytuł,Ukształtowanie działki,Winda,CITY,LAT,LON
str,date,date,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64
"""dzialka-pupki""",2025-11-22,2026-01-31,94000.0,50.0,null,"""70,00 m""",null,"""46804/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-pupki/""",null,null,"""Brak""",null,"""Brak planu miejscowego""",null,1880.0,null,"""Budowlana""",null,null,null,null,"""28,00 m""",null,null,"""Na sprzedaż działki budowlane w Pupkach, gm.Turośl""","""Płaska""",null,"""PUPKI""",53.2258,21.8497
"""dzialka-stare-kupiski-ul-wspolna""",2025-11-22,2025-12-02,129000.0,162.88,null,"""29,00 m""",null,"""46747/3877/OGS""","""Kwadrat""","""https://polnoc.pl/oferty/dzialka-stare-kupiski-ul-wspolna/""",null,null,"""Brak""",null,"""Brak planu miejscowego""",null,792.0,null,"""Budowlana""",null,null,null,null,"""28,00 m""",null,null,"""Działka budowlana z WZ w Starych Kupiskach!""","""Płaska""",null,"""STARE KUPISKI""",53.1972,21.9964
"""dzialka-lomza-zdrojowa""",2025-11-22,2026-01-31,249000.0,214.47,null,null,null,"""46675/3877/OGS""","""Trapez""","""https://polnoc.pl/oferty/dzialka-lomza-zdrojowa/""",null,null,"""Mieszane""",null,"""Brak planu miejscowego""",null,1161.0,null,"""Budowlana""",null,null,null,null,null,null,null,"""Działka w Łomży na ul. Zdrojowej!""","""Zróżnicowana""",null,"""ŁOMŻA""",53.1781,22.0592
"""dzialka-lomza-waska""",2025-11-22,2026-01-31,205000.0,272.97,null,"""26,00 m""",null,"""44481/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-lomza-waska/""",null,null,"""Brak""",null,"""Brak planu miejscowego""",null,751.0,null,"""Budowlana""",null,null,null,null,"""29,00 m""",null,null,"""Malownicza działka w Łomży na ul. Wąskiej!""","""Płaska""",null,"""ŁOMŻA""",53.1781,22.0592
"""dzialka-lomza-grabowa""",2025-11-22,2026-01-31,239000.0,247.02,null,"""32,00 m""",null,"""46667/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-lomza-grabowa/""",null,null,"""Brak""",null,"""Brak planu miejscowego""",null,1008.0,null,"""Budowlana""",null,null,null,null,"""33,50 m""",null,null,"""Przestronne działki w Łomży na ul. Grabowej!""","""Płaska""",null,"""ŁOMŻA""",53.1781,22.0592
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""dzialka-lomza-ul-dzialkowa""",2025-12-31,2026-02-14,370000.0,430.23,null,null,null,"""46948/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-lomza-ul-dzialkowa/""",null,null,"""Brak""",null,"""Plan zagospodarowania przestrzennego""",null,860.0,null,"""Budowlana""",null,null,null,null,null,null,null,"""Działka – Łomża, ul. Działkowa""","""Płaska""",null,"""ŁOMŻA""",53.1781,22.0592
"""dzialka-lomza-ul-dzialkowa-2""",2026-01-09,2026-01-31,370000.0,430.23,null,null,null,"""46948/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-lomza-ul-dzialkowa-2/""",null,null,"""Brak""",null,"""Plan zagospodarowania przestrzennego""",null,860.0,null,"""Budowlana""",null,null,null,null,null,null,null,"""Działka – Łomża, ul. Działkowa""","""Płaska""",null,"""ŁOMŻA""",53.1781,22.0592
"""dzialka-budy-czarnockie-2""",2026-01-09,2026-01-31,105000.0,106.49,null,"""44,50 m""",null,"""47008/3877/OGS""","""Prostokąt""","""https://polnoc.pl/oferty/dzialka-budy-czarnockie-2/""",null,null,"""Brak""",null,"""Brak planu miejscowego""",null,986.0,null,"""Budowlana""",null,null,null,null,"""22,00 m""",null,null,"""Atrakcyjna działka w Budach Czarnockich!""","""Płaska""",null,"""BUDY CZARNOCKIE""",53.2167,22.15


# 4LOMZA

In [61]:
data["analyzed_offers"] = data["analyzed_offers"].filter((pl.col("ESTATE_TYPE").str.contains("DZIAŁKA BUDOWLANA"))&
                               (pl.col("OFFER_TYPE").str.contains("SPRZEDAZ")))

In [62]:
data["analyzed_offers"] = data["analyzed_offers"].unique(subset=["ID"])

In [63]:
data["analyzed_offers"] = data["analyzed_offers"].with_columns(
    pl.col(["LAST_UPDATED", "DATE_ADDED"]).str.to_date(format="%Y-%m-%d")
            )

In [64]:
data["analyzed_offers"] = data["analyzed_offers"].with_columns(
    pl.col([ "ANNOUNCE_DATE"]).str.to_date(format="%Y-%m-%d")
            )

In [65]:
data["analyzed_offers"] = data["analyzed_offers"].with_columns(
    pl.col("AREA_M2")
    .str.replace(" m2", "")
    .str.replace(",", ".")
    .cast(pl.Float64, strict=False)  
)

In [66]:
data["analyzed_offers"] = data["analyzed_offers"].with_columns(
    pl.col("PRICE")
    .str.replace(" m2", "")
    .str.replace(",", ".")
    .cast(pl.Float64, strict=False)  
)

In [67]:
data["analyzed_offers"]

ID,DATE_ADDED,LAST_UPDATED,ANNOUNCE_DATE,TEXT,ANALYSIS_RESULT,RAW_ARGS,OFFER_TYPE,ESTATE_TYPE,PRICE,AREA_M2,LOCATION
str,date,date,date,str,str,str,str,str,f64,f64,str
"""sprzedam działkę budowlaną w_2025-12-27""",2025-12-28,2026-01-03,2025-12-27,"""Sprzedam działkę budowlaną w Łomży ul. Piaski pow. 701 m2, wydane warunki zabudowy na budynek mieszkalny jednorodzinny.""","""{""name"": ""parse_offer"", ""arguments"": ""{\""OFFER_TYPE\"": \""SPRZEDAZ\"", \""ESTATE_TYPE\"": \""DZIAŁKA BUDOWLANA\"", \""PRICE\"": \""BRAK\"", \""AREA_M2\"": \""701\"", \""LOCATION\"": \""Łomża, ul. Piaski\""}""} tool {""parse_offer"": ""{\""name\"": \""parse_offer\"", \""arguments\"": {\""OFFER_TYPE\"": \""SPRZEDAZ\"", \""ESTATE_TYPE\"": \""DZIAŁKA BUDOWLANA\"", \""PRICE\"": \""BRAK\"", \""AREA_M2\"": \""701\"", \""LOCATION\"": \""Łomża, ul. Piaski\""""","""OFFER_TYPE: SPRZEDAZ, ESTATE_TYPE: DZIAŁKA BUDOWLANA, PRICE: BRAK, AREA_M2: 701, LOCATION: ŁOMŻA, UL. PIASKI""","""SPRZEDAZ""","""DZIAŁKA BUDOWLANA""",null,701.0,"""ŁOMŻA"""
"""sprzedam działkę budowlaną 10_2026-01-19""",2026-01-20,2026-01-26,2026-01-19,"""Sprzedam działkę budowlaną 10 a ul Konarska, Zawady.""","""{""name"": ""parse_offer"", ""arguments"": ""{\""offer_type\"": \""SPRZEDAZ\"", \""estate_type\"": \""DZIAŁKA BUDOWLANA\"", \""price\"": \""BRAK\"", \""area_m2\"": \""BRAK\"", \""location\"": \""Konarska, Zawady\""}""} tool {""parse_offer"": ""{\""name\"": \""parse_offer\"", \""arguments\"": {\""offer_type\"": \""SPRZEDAZ\"", \""estate_type\"": \""DZIAŁKA BUDOWLANA\"", \""price\"": \""BRAK\"", \""area_m2\"": \""BRAK\"", \""location\"": \""Konarska, Zawady\""}, \""output\"": {\""name\"": \""parse_of""","""OFFER_TYPE: SPRZEDAZ, ESTATE_TYPE: DZIAŁKA BUDOWLANA, PRICE: BRAK, AREA_M2: BRAK, LOCATION: KONARSKA, ZAWADY""","""SPRZEDAZ""","""DZIAŁKA BUDOWLANA""",null,null,"""KONARSKA"""
"""sprzedam działkę z bezterminowymi_2026-02-20""",2026-02-20,2026-02-23,2026-02-20,"""Sprzedam działkę z bezterminowymi warunkami zabudowy w Nowogrodzie, pow. 1275 m2. Cena 115 000 zł.""","""{""name"": ""parse_offer"", ""arguments"": ""{\""offer_type\"": \""SPRZEDAZ\"", \""estate_type\"": \""DZIAŁKA BUDOWLANA\"", \""price\"": 115000, \""area_m2\"": 1275, \""location\"": \""Nowogród\""}""} tool {""parse_offer"": ""{\""name\"": \""parse_offer\"", \""arguments\"": {\""offer_type\"": \""SPRZEDAZ\"", \""estate_type\"": \""DZIAŁKA BUDOWLANA\"", \""price\"": 115000, \""area_m2\"": 1275, \""location\"": \""Nowogród\""}, \""output\"": {\""name\"": \""parse_offer\"", \""arguments""","""OFFER_TYPE: SPRZEDAZ, ESTATE_TYPE: DZIAŁKA BUDOWLANA, PRICE: 115000, AREA_M2: 1275, LOCATION: NOWOGRÓD""","""SPRZEDAZ""","""DZIAŁKA BUDOWLANA""",115000.0,1275.0,"""NOWOGRÓD"""
"""sprzedam dzialke 8 ary_2025-11-28""",2025-11-29,2025-12-04,2025-11-28,"""Sprzedam dzialke 8 ary z warunkami zabudowy 2 siewniki zbiornik do mleka 450 lplug 4 okolice lomzy nie odpowiadam na esemesy.""","""{""name"": ""parse_offer"", ""arguments"": ""{\""offer_type\"": \""SPRZEDAZ\"", \""estate_type\"": \""DZIAŁKA BUDOWLANA\"", \""price\"": \""BRAK\"", \""area_m2\"": \""BRAK\"", \""location\"": \""Lomza\""}""} tool {""parse_offer"": ""{\""name\"": \""parse_offer\"", \""arguments\"": {\""offer_type\"": \""SPRZEDAZ\"", \""estate_type\"": \""DZIAŁKA BUDOWLANA\"", \""price\"": \""BRAK\"", \""area_m2\"": \""BRAK\"", \""location\"": \""Lomza\""}, \""output\"": {\""name\"": \""parse_offer\"", \""arguments""","""OFFER_TYPE: SPRZEDAZ, ESTATE_TYPE: DZIAŁKA BUDOWLANA, PRICE: BRAK, AREA_M2: BRAK, LOCATION: LOMZA""","""SPRZEDAZ""","""DZIAŁKA BUDOWLANA""",null,null,"""LOMZA"""
"""sprzedam działkę budowlaną nr_2025-11-30""",2025-12-01,2025-12-06,2025-11-30,"""Sprzedam działkę budowlaną nr 817/32 (pow.1364m2) w Nowogrodzie z warunkami zabudowy za 160tyś.zł. Działka z pięknym widokiem na dolinę rzeki Narwi""","""{""name"": ""parse_offer"", ""arguments"": ""{\""offer_type\"": \""SPRZEDAZ\"", \""estate_type\"": \""DZIAŁKA BUDOWLANA\"", \""price\"": 1600000, \""area_m2\"": 1364, \""locati

In [68]:
stats = data["analyzed_offers"].select([
    pl.col("AREA_M2").quantile(0.25).alias("q1"),
    pl.col("AREA_M2").quantile(0.75).alias("q3")
])

q1 = stats[0, "q1"]
q3 = stats[0, "q3"]
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"Statystyczne granice dla AREA_M2: {lower_bound:.2f} - {upper_bound:.2f}")

outliery_statystyczne = data["analyzed_offers"].filter(
    (pl.col("AREA_M2") < lower_bound) | (pl.col("AREA_M2") > upper_bound)
)

Statystyczne granice dla AREA_M2: -350.00 - 2450.00


In [69]:
corrected_outliers = outliery_statystyczne.with_columns(
    pl.when(
        (pl.col("TEXT").str.contains(f"(?i)ar")) | 
        (pl.col("TEXT").str.contains(f"(?i)ara"))   
    )
    .then(pl.col("AREA_M2") / 10)  
    .otherwise(pl.col("AREA_M2"))
    .alias("AREA_M2_CORRECTED")
).select(["ID", "AREA_M2_CORRECTED"])

In [70]:
data["analyzed_offers"] = data["analyzed_offers"].join(
    corrected_outliers, on="ID", how="left"
)

data["analyzed_offers"] = data["analyzed_offers"].with_columns(
    pl.col("AREA_M2_CORRECTED").fill_null(pl.col("AREA_M2")).alias("AREA_M2")
).drop("AREA_M2_CORRECTED")

In [71]:
data["analyzed_offers"] = data["analyzed_offers"].with_columns(
    (pl.col("PRICE") / pl.col("AREA_M2")).round(2).alias("PRICE_M2")
)

In [72]:
data["analyzed_offers"] = data["analyzed_offers"].rename({"LOCATION":"CITY"})

In [73]:
data["analyzed_offers"] = data["analyzed_offers"].with_columns(
    pl.col("CITY")
    .str.to_uppercase()
    .str.replace_all("Ł", "L")
    .str.replace_all("Ó", "O")
    .str.replace_all("Ś", "S")
    .str.replace_all("Ż", "Z")
    .str.replace_all("Ź", "Z")
    .str.replace_all("Ć", "C")
    .str.replace_all("Ń", "N")
    .str.replace_all("Ą", "A")
    .str.replace_all("Ę", "E")
    .alias("SEARCH_CITY")
)

condition = pl.when(False).then(None)

for original_city in CITIES_GEO.keys():
    simple_city = simplify_text(original_city) 
    
    condition = condition.when(
        pl.col("SEARCH_CITY").str.contains(simple_city)
    ).then(pl.lit(original_city)) 

data["analyzed_offers"] = data["analyzed_offers"].with_columns(
    condition.alias("CLEAN_CITY")
)

data["analyzed_offers"] = data["analyzed_offers"].drop("SEARCH_CITY")

In [74]:
data["analyzed_offers"].filter(pl.col("CLEAN_CITY").is_null())["CITY"].unique().to_list()

['POZNAŃSKA',
 'WALKOWA',
 'KONARSKA',
 'ELBLĄSKA',
 '12KM OD ŁOMŻY',
 '8 KM OD ŁOMŻY',
 'BRAK',
 'UL SŁONECZNA',
 'KRUCZA',
 'UL.DMOWSKIEGO',
 'MIASTKÓW']

In [75]:
data["analyzed_offers"] = data["analyzed_offers"].with_columns(
    pl.when(pl.col("CITY").str.to_uppercase() == "BRAK")
    .then(None) 
    .when(pl.col("CLEAN_CITY").is_null()) 
    .then(pl.lit("ŁOMŻA")) 
    .otherwise(pl.col("CLEAN_CITY")) 
    .alias("CLEAN_CITY")
)

In [76]:
data["analyzed_offers"] = data["analyzed_offers"].drop( ["CITY"])
data["analyzed_offers"] = data["analyzed_offers"].rename({"CLEAN_CITY":"CITY"})
data["analyzed_offers"] = data["analyzed_offers"].join(geo_df, on="CITY", how="left")

In [77]:
data.keys()

dict_keys(['offers', 'offers_tytan', 'offers_gralczyk', 'offers_polnoc', 'analyzed_offers'])

# Concating all datasets

In [78]:
for key in data.keys():
    print (key, data[key].columns)

offers ['ID', 'TEXT', 'LINK', 'VALUE', 'CURRENCY', 'NEGOTIABLE', 'AREA_M2', 'PRICE_M2', 'CITY', 'DATE_ORIG', 'LAST_UPDATED', 'DATE_ADDED', 'LAT', 'LON', 'DAY_NAME_EN']
offers_tytan ['ID', 'CITY', 'NAME', 'AREA_M2', 'PRICE', 'PRICE_M2', 'LINK', 'LAST_UPDATED', 'DATE_ADDED', 'DAY_NAME_EN', 'LAT', 'LON']
offers_gralczyk ['ID', 'DATE_ADDED', 'PRICE', 'PRICE_M2', 'Długość', 'Forma własności', 'Gaz', 'Kanalizacja', 'LAST_UPDATED', 'LINK', 'Liczba linii tel.', 'Numer oferty', 'AREA_M2', 'Powierzchnia zabudowy', 'Przeznaczenie', 'Prąd', 'Szerokość', 'Sąsiedztwo', 'Typ drogi', 'Typ działki', 'Typ ogrodzenia', 'Typ transakcji', 'Woda', 'Wygląd działki', 'Wysokość min.', 'Łącze internetowe', 'CITY', 'LAT', 'LON']
offers_polnoc ['ID', 'DATE_ADDED', 'LAST_UPDATED', 'PRICE', 'PRICE_M2', 'Ciepła woda', 'Długość działki (mb.)', 'Garaż/Miejsca parkingowe', 'Id oferty', 'Kształt działki', 'LINK', 'Liczba pięter', 'Liczba pokoi', 'Ogrodzenie', 'Piętro', 'Plan miejscowy', 'Powierzchnia', 'AREA_M2', 'Powie

In [80]:
target_cols = ["ID", "DATE_ADDED", "LAST_UPDATED", "AREA_M2", "PRICE_M2", "CITY", "LAT", "LON", "PRICE"]

In [82]:
processed_dfs = []

for key, df in data.items():
    temp_df = df.rename({"VALUE": "PRICE"}) if "VALUE" in df.columns else df
    temp_df = temp_df.select(target_cols).with_columns([
        pl.col("ID").cast(pl.String),        
        pl.col("PRICE").cast(pl.Float64),  
        pl.col("AREA_M2").cast(pl.Float64),
        pl.col("PRICE_M2").cast(pl.Float64),
        pl.col("LAT").cast(pl.Float64),
        pl.col("LON").cast(pl.Float64),
        pl.lit(key).alias("SOURCE")
    ])
    processed_dfs.append(temp_df)

data["Offers_all"] = pl.concat(processed_dfs)

In [97]:
data["Offers_all"] = data["Offers_all"].with_columns([
    pl.when(pl.col("PRICE") < 15_000)
    .then(None)
    .otherwise(pl.col("PRICE"))
    .alias("PRICE"),
]).with_columns([
    pl.when(pl.col("PRICE").is_null())
    .then(None)
    .otherwise(pl.col("PRICE_M2"))
    .alias("PRICE_M2")
])

In [104]:
data["Offers_all"]

ID,DATE_ADDED,LAST_UPDATED,AREA_M2,PRICE_M2,CITY,LAT,LON,PRICE,SOURCE
str,date,date,f64,f64,str,f64,f64,f64,str
"""Łomża_2134_99000""",2025-11-10,2025-11-11,2134.0,46.39,"""ŁOMŻA""",53.1781,22.0592,99000.0,"""offers"""
"""Zawady_1000_130000""",2026-02-11,2025-11-11,1000.0,130.0,"""ZAWADY""",53.1539,22.6653,130000.0,"""offers"""
"""Elżbiecin_12200_1490000""",2026-02-10,2025-11-11,12200.0,122.13,"""ELŻBIECIN""",53.1672,22.1231,1.49e6,"""offers"""
"""Jurki_3000_67500""",2025-11-06,2025-11-11,3000.0,22.5,"""JURKI""",53.3983,22.1114,67500.0,"""offers"""
"""Łomża_1000_129500""",2025-11-06,2025-11-11,1000.0,129.5,"""ŁOMŻA""",53.1781,22.0592,129500.0,"""offers"""
…,…,…,…,…,…,…,…,…,…
"""na sprzedaż działka budowlana_2026-02-05""",2026-02-06,2026-02-12,1054.0,null,"""NOWOGRÓD""",53.2264,21.8806,null,"""analyzed_offers"""
"""sprzedam działkę budowlaną w_2026-02-17""",2026-02-18,2026-02-18,null,null,"""NOWOGRÓD""",53.2264,21.8806,null,"""analyzed_offers"""
"""na sprzedaż atrakcyjna działka_2026-02-05""",2026-02-06,2026-02-12,505.0,null,"""ŁOMŻA""",53.1781,22.0592,null,"""analyzed_offers"""


## Day of the week mapping

In [106]:
week_days = {
    1: "Poniedziałek",
    2: "Wtorek",
    3: "Środa",
    4: "Czwartek",
    5: "Piątek",
    6: "Sobota",
    7: "Niedziela"
}

In [108]:
data["Offers_all"] = data["Offers_all"].with_columns([
    pl.col("DATE_ADDED").cast(pl.Date),
    pl.col("DATE_ADDED")
    .dt.weekday()
    .cast(pl.String) 
    .replace(week_days)
    .alias("DAY_NAME_PL")
])

## Cleaning AREA_M2

In [110]:
data["Offers_all"] = data["Offers_all"].with_columns(
    pl.when(pl.col("AREA_M2") <= 1.0)
    .then(None)
    .otherwise(pl.col("AREA_M2"))
    .alias("AREA_M2")
)

In [113]:
data["Offers_all"] = data["Offers_all"].with_columns([
    pl.when(pl.col("AREA_M2") < 100.0)
    .then(pl.col("AREA_M2") * 100.0)
    .otherwise(pl.col("AREA_M2"))
    .alias("AREA_M2")
]).with_columns([
    pl.when(pl.col("AREA_M2") < 100.0)
    .then(None)
    .otherwise(pl.col("AREA_M2"))
    .alias("AREA_M2")
]).with_columns([
    pl.when((pl.col("PRICE").is_not_null()) & (pl.col("AREA_M2").is_not_null()))
    .then(pl.col("PRICE") / pl.col("AREA_M2"))
    .otherwise(None)
    .alias("PRICE_M2")
])
print(data["Offers_all"].filter(pl.col("ID").str.contains("MORGOWNIKI|NOWOGRÓD|Zawady_10")).select([
    "ID", "AREA_M2", "PRICE", "PRICE_M2"
]))

shape: (2, 4)
┌────────────────────┬─────────┬──────────┬──────────┐
│ ID                 ┆ AREA_M2 ┆ PRICE    ┆ PRICE_M2 │
│ ---                ┆ ---     ┆ ---      ┆ ---      │
│ str                ┆ f64     ┆ f64      ┆ f64      │
╞════════════════════╪═════════╪══════════╪══════════╡
│ Zawady_1000_130000 ┆ 1000.0  ┆ 130000.0 ┆ 130.0    │
│ Zawady_10_89000    ┆ 1000.0  ┆ 89000.0  ┆ 89.0     │
└────────────────────┴─────────┴──────────┴──────────┘


In [119]:
data["Offers_all"] = data["Offers_all"].with_columns(
    pl.when(pl.col("ID") == "dzialka-jedwabne-lomzynska")
    .then(pl.lit(10200.0))
    .otherwise(pl.col("AREA_M2"))
    .alias("AREA_M2")
)

data["Offers_all"] = data["Offers_all"].with_columns(
    pl.when(pl.col("ID") == "dzialka-jedwabne-lomzynska")
    .then(pl.col("PRICE") / pl.col("AREA_M2"))
    .otherwise(pl.col("PRICE_M2"))
    .alias("PRICE_M2")
)

In [120]:
data["Offers_all"]

ID,DATE_ADDED,LAST_UPDATED,AREA_M2,PRICE_M2,CITY,LAT,LON,PRICE,SOURCE,DAY_NAME_PL
str,date,date,f64,f64,str,f64,f64,f64,str,str
"""Łomża_2134_99000""",2025-11-10,2025-11-11,2134.0,46.391753,"""ŁOMŻA""",53.1781,22.0592,99000.0,"""offers""","""Poniedziałek"""
"""Zawady_1000_130000""",2026-02-11,2025-11-11,1000.0,130.0,"""ZAWADY""",53.1539,22.6653,130000.0,"""offers""","""Środa"""
"""Elżbiecin_12200_1490000""",2026-02-10,2025-11-11,12200.0,122.131148,"""ELŻBIECIN""",53.1672,22.1231,1.49e6,"""offers""","""Wtorek"""
"""Jurki_3000_67500""",2025-11-06,2025-11-11,3000.0,22.5,"""JURKI""",53.3983,22.1114,67500.0,"""offers""","""Czwartek"""
"""Łomża_1000_129500""",2025-11-06,2025-11-11,1000.0,129.5,"""ŁOMŻA""",53.1781,22.0592,129500.0,"""offers""","""Czwartek"""
…,…,…,…,…,…,…,…,…,…,…
"""na sprzedaż działka budowlana_2026-02-05""",2026-02-06,2026-02-12,1054.0,null,"""NOWOGRÓD""",53.2264,21.8806,null,"""analyzed_offers""","""Piątek"""
"""sprzedam działkę budowlaną w_2026-02-17""",2026-02-18,2026-02-18,null,null,"""NOWOGRÓD""",53.2264,21.8806,null,"""analyzed_offers""","""Środa"""
"""na sprzedaż atrakcyjna działka_2026-02-05""",2026-02-06,2026-02-12,505.0,null,"""ŁOMŻA""",53.1781,22.0592,null,"""analyzed_offers""","""Piątek"""


## Measuring distance from main_city

In [136]:
LOMZA_LAT = 53.1781
LOMZA_LON = 22.0592
data["Offers_all"] = add_haversine_distance(data["Offers_all"], LOMZA_LAT, LOMZA_LON)

In [137]:
data["Offers_all"]

ID,DATE_ADDED,LAST_UPDATED,AREA_M2,PRICE_M2,CITY,LAT,LON,PRICE,SOURCE,DAY_NAME_PL,MAIN_CITY_DIST
str,date,date,f64,f64,str,f64,f64,f64,str,str,f64
"""Łomża_2134_99000""",2025-11-10,2025-11-11,2134.0,46.391753,"""ŁOMŻA""",53.1781,22.0592,99000.0,"""offers""","""Poniedziałek""",0.0
"""Zawady_1000_130000""",2026-02-11,2025-11-11,1000.0,130.0,"""ZAWADY""",53.1539,22.6653,130000.0,"""offers""","""Środa""",40.492747
"""Elżbiecin_12200_1490000""",2026-02-10,2025-11-11,12200.0,122.131148,"""ELŻBIECIN""",53.1672,22.1231,1.49e6,"""offers""","""Wtorek""",4.428093
"""Jurki_3000_67500""",2025-11-06,2025-11-11,3000.0,22.5,"""JURKI""",53.3983,22.1114,67500.0,"""offers""","""Czwartek""",24.729752
"""Łomża_1000_129500""",2025-11-06,2025-11-11,1000.0,129.5,"""ŁOMŻA""",53.1781,22.0592,129500.0,"""offers""","""Czwartek""",0.0
…,…,…,…,…,…,…,…,…,…,…,…
"""na sprzedaż działka budowlana_2026-02-05""",2026-02-06,2026-02-12,1054.0,null,"""NOWOGRÓD""",53.2264,21.8806,null,"""analyzed_offers""","""Piątek""",13.05184
"""sprzedam działkę budowlaną w_2026-02-17""",2026-02-18,2026-02-18,null,null,"""NOWOGRÓD""",53.2264,21.8806,null,"""analyzed_offers""","""Środa""",13.05184
"""na sprzedaż atrakcyjna działka_2026-02-05""",2026-02-06,2026-02-12,505.0,null,"""ŁOMŻA""",53.1781,22.0592,null,"""analyzed_offers""","""Piątek""",0.0


## Size segments

In [139]:
size_bins = [700.0, 1000.0, 1500.0, 5000.0, 10000.0]

size_labels = [
    "Tiny/Sub-standard",    
    "Small Plot",           
    "Medium Plot",         
    "Large Plot",           
    "Investment/Small Farm",
    "Hectares/Agriculture" 
]

data["Offers_all"] = data["Offers_all"].with_columns(
    pl.col("AREA_M2").cut(
        size_bins, 
        labels=size_labels
    ).alias("SIZE_SEGMENT")
)

In [140]:
data["Offers_all"]

ID,DATE_ADDED,LAST_UPDATED,AREA_M2,PRICE_M2,CITY,LAT,LON,PRICE,SOURCE,DAY_NAME_PL,MAIN_CITY_DIST,SIZE_SEGMENT
str,date,date,f64,f64,str,f64,f64,f64,str,str,f64,cat
"""Łomża_2134_99000""",2025-11-10,2025-11-11,2134.0,46.391753,"""ŁOMŻA""",53.1781,22.0592,99000.0,"""offers""","""Poniedziałek""",0.0,"""Large Plot"""
"""Zawady_1000_130000""",2026-02-11,2025-11-11,1000.0,130.0,"""ZAWADY""",53.1539,22.6653,130000.0,"""offers""","""Środa""",40.492747,"""Small Plot"""
"""Elżbiecin_12200_1490000""",2026-02-10,2025-11-11,12200.0,122.131148,"""ELŻBIECIN""",53.1672,22.1231,1.49e6,"""offers""","""Wtorek""",4.428093,"""Hectares/Agriculture"""
"""Jurki_3000_67500""",2025-11-06,2025-11-11,3000.0,22.5,"""JURKI""",53.3983,22.1114,67500.0,"""offers""","""Czwartek""",24.729752,"""Large Plot"""
"""Łomża_1000_129500""",2025-11-06,2025-11-11,1000.0,129.5,"""ŁOMŻA""",53.1781,22.0592,129500.0,"""offers""","""Czwartek""",0.0,"""Small Plot"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""na sprzedaż działka budowlana_2026-02-05""",2026-02-06,2026-02-12,1054.0,null,"""NOWOGRÓD""",53.2264,21.8806,null,"""analyzed_offers""","""Piątek""",13.05184,"""Medium Plot"""
"""sprzedam działkę budowlaną w_2026-02-17""",2026-02-18,2026-02-18,null,null,"""NOWOGRÓD""",53.2264,21.8806,null,"""analyzed_offers""","""Środa""",13.05184,null
"""na sprzedaż atrakcyjna działka_2026-02-05""",2026-02-06,2026-02-12,505.0,null,"""ŁOMŻA""",53.1781,22.0592,null,"""analyzed_offers""","""Piątek""",0.0,"""Tiny/Sub-standard"""
